In [1]:
import os
import sys
dir_path = "/qumulo/shared_data/aofei_summer/RegTok/RegLLM"
sys.path.insert(0, dir_path)
os.environ['CUDA_VISIBLE_DEVICES'] = "1"
os.environ["HF_HUB_CACHE"]="/qumulo/shared_data/aofei_summer/LLMs"
from llava.eval.cli_v1 import RegLLMChatbot

/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [6]:
model_dir = "/qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/reg_seg_instruct_mix_45k"
model_args = {
        "model_name_or_path": "Qwen/Qwen3-8B",
        "pretrained_llm_path": model_dir,
        "tokenizer_path": None,
        "peft_path": None,
        "regtok_config_path": "/qumulo/shared_data/aofei_summer/RegTok/source/tokenizer/regtok_config.yaml",
        "regtok_weight_path": "/qumulo/shared_data/aofei_summer/intern_records/RegTok/checkpoints/RegTok_pipeline_full_wo_quant/002-RegTok/checkpoints/0079280.pt",
        "use_regtok": True,
        "mm_vision_vq_type": "RegTok",
        "use_region_tokens": False,
        "vision_tower": "/qumulo/shared_data/aofei_summer/CLIPs/unimed_clip_vit_l14.pt",
        "mm_use_im_start_end": False,
        "mm_use_im_patch_token": True,
        "mm_vision_select_feature": "patch",
        "mm_patch_merge_type": "flat",
        "mm_projector_type": "mlp2x_gelu",
        "pretrain_mm_mlp_adapter": None,
        "mm_vision_select_layer": -1,
        "use_region_tokens": True,
        "use_sep_proj": True,
        "output_segmentation": True,
        "use_seg_loss": True,
        "modality_num": 18,
        "codebook_size": 32,
        "train_all_embeddings": True
    }

In [7]:
bot = RegLLMChatbot(model_dir, model_args=model_args, device="cuda")

loading model from /qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/reg_seg_instruct_mix_45k
use RegSegForCausalLM!
576 codebook_token_ids


Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 24.61it/s]
Some weights of the model checkpoint at /qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/reg_seg_instruct_mix_45k were not used when initializing RegSegForCausalLM: ['model.region_mm_projector.0.bias', 'model.region_mm_projector.0.weight', 'model.region_mm_projector.2.bias', 'model.region_mm_projector.2.weight', 'model.vision_tower.vision_tower.image_encoder.class_embedding', 'model.vision_tower.vision_tower.image_encoder.conv1.weight', 'model.vision_tower.vision_tower.image_encoder.ln_post.bias', 'model.vision_tower.vision_tower.image_encoder.ln_post.weight', 'model.vision_tower.vision_tower.image_encoder.ln_pre.bias', 'model.vision_tower.vision_tower.image_encoder.ln_pre.weight', 'model.vision_tower.vision_tower.image_encoder.positional_embedding', 'model.vision_tower.vision_tower.image_encoder.proj', 'model.vision_tower.vision_tower.image_encoder.transformer.resblocks.0.attn.in_proj_bias', 'mode

load vision tower!
Number of stacks: 1
Upsample mode: conv
tokenflow load from: /qumulo/shared_data/aofei_summer/intern_records/RegTok/checkpoints/RegTok_pipeline_full_wo_quant/002-RegTok/checkpoints/0079280.pt
tokenflow model load success!!
pre loading complete!
Loading tokenizer from /qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/reg_seg_instruct_mix_45k


In [4]:
# bot = RegLLMChatbot(model_dir, model_args=model_args, device="cuda")

In [9]:
bot.inference("Segment the abnormal region in the image.", images="/qumulo/shared_data/aofei_summer/data/evaluation/imgs/xmlab102/source.jpg")

['assistant\nThe abnormal region is segmented as [M5_29], located at bbox [0.264, 0.318, 0.157, 0.192].']

In [ ]:
bot.generate()

In [15]:
import json
from tqdm import tqdm
model_name = "instruct_regseg_83k"
dataset_name = "VQA-RAD"
# dataset_name = "OmniMed"
modality_name = "CT"
# question_file = "/qumulo/shared_data/aofei_summer/data/evaluation/test_instruct2.json"
# answers_file = f"/qumulo/shared_data/aofei_summer/data/evaluation/inference/answers_{model_name}.jsonl"
# image_folder = "/qumulo/shared_data/aofei_summer/data/evaluation/imgs"

question_file = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/test_instruct.json"
answers_file = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/inference/answers_{model_name}_ins.jsonl"
image_folder = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/image"


# question_file = f"/qumulo/shared_data/aofei_summer/data/OmniMed/OmniMedVQA/OmniMedVQA/QA_information/Open-access/Modality_CT(Computed Tomography).json"
# answers_file = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/inference/answers_{modality_name}_{model_name}.jsonl"
# image_folder = f"/qumulo/shared_data/aofei_summer/data/OmniMed/OmniMedVQA/OmniMedVQA"


In [16]:
questions = json.load(open(os.path.expanduser(question_file), "r"))
answers_file = os.path.expanduser(answers_file)
os.makedirs(os.path.dirname(answers_file), exist_ok=True)
ans_file = open(answers_file, "w")
for line in tqdm(questions):

    # idx = line["qid"]
    # question = line["question"] # ['value'].split('\n')[0]
    # gt_ans = line["answer"] # ['value']      
    # image_file = line["img_name"]

    # idx = line["question_id"]
    # question = line["question"] + ". Answer this question shortly by selecting one option." # ['value'].split('\n')[0]
    # gt_ans = line["gt_answer"] # ['value']      
    # image_file = line["image_path"]

    idx = line["id"]
    question = line["conversations"][0]["value"] # ['value'].split('\n')[0]
    gt_ans = line['conversations'][1]['value'] # ['value']
    image_file = line["image"]

    qs = question
    
    image_file = os.path.join(image_folder, image_file)
    ans = bot.inference(qs, image_file)[0]
    ans = ans.replace("assistant\n", "").strip()

    ans_file.write(json.dumps({"question_id": idx,
                                   "prompt": qs,
                                   "text": ans,
                                   "gt_ans": gt_ans,
                                   "metadata": {}}) + "\n")
    ans_file.flush()
ans_file.close()

  0%|          | 0/451 [00:00<?, ?it/s]

100%|██████████| 451/451 [08:56<00:00,  1.19s/it]


In [14]:
# read_answer_file jsonlines
answers = []
with open(answers_file, 'r') as file:
    answers = [json.loads(line) for line in file]

NameError: name 'answers_file' is not defined

In [ ]:
questions = json.load(open(os.path.expanduser(question_file), "r"))
answers_file = os.path.expanduser(answers_file)
os.makedirs(os.path.dirname(answers_file), exist_ok=True)
ans_file = open(answers_file, "w")
for line in tqdm(questions):

    idx = line["qid"]
    question = line["question"] # ['value'].split('\n')[0]
    gt_ans = line["answer"] # ['value']      
    image_file = line["img_name"]

    # idx = line["id"]
    # question = line["conversations"][0]["value"] # ['value'].split('\n')[0]
    # gt_ans = line['conversations'][1]['value'] # ['value']
    # image_file = line["image"]

    qs = question
    
    image_file = os.path.join(image_folder, image_file)
    ans = bot.inference(qs, image_file)[0]
    ans = ans.replace("assistant\n", "").strip()

    ans_file.write(json.dumps({"question_id": idx,
                                   "prompt": qs,
                                   "text": ans,
                                   "gt_ans": gt_ans,
                                   "metadata": {}}) + "\n")
    ans_file.flush()
ans_file.close()

100%|██████████| 1061/1061 [45:51<00:00,  2.59s/it]


In [9]:
(79.92 + 47.16) / 2

63.54

In [3]:
from eval_metrics.glossary import normalize_word

from eval_metrics.utils import split_sentence

def calculate_f1score(candidate, reference):

    candidate = normalize_word(candidate)
    reference = normalize_word(reference)

    candidate_words = split_sentence(candidate, 1)
    reference_words = split_sentence(reference, 1)
    word_set = set()
    for word in candidate_words:
        word_set.add(word)
    for word in reference_words:
        word_set.add(word)
    
    tp = 0
    fp = 0
    fn = 0
    for word in word_set:
        if word in candidate_words and word in reference_words:
            tp += candidate_words[word]
        elif word in candidate_words and word not in reference_words:
            fp += candidate_words[word]
        elif word not in candidate_words and word in reference_words:
            fn += reference_words[word]
    
    if len(candidate_words) == 0:
        return 0, 0, 0 # "0 (warning: length of candidate's words is 0)"
    elif len(reference_words) == 0:
        return 0, 0, 0
    else:
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        if tp == 0:
            return 0, 0, 0
        else:
            return 2 * precision * recall / (precision + recall), precision, recall

    # return candidate_words, reference_words, word_set

In [13]:
calculate_f1score("Two organs (2) are visible: the lungs", "2")

(0.5, 0.3333333333333333, 1.0)

In [ ]:
""

In [ ]:
def preprocess_answer(answer):
    # words to replace: lungs -> lung, abdominal -> abdomen, 
    # remove words like \u2014 